# 💻 Unidad 1: Material Complementario - Práctica
## Módulo 04 - Estadística Descriptiva Avanzada
### Laboratorio - Universidad del Aconcagua

---

## 🎯 Objetivos

1. ✅ Calcular asimetría, curtosis y percentiles
2. ✅ Aplicar estadística robusta (MAD, mediana)
3. ✅ Testear normalidad con Shapiro-Wilk y KS
4. ✅ Implementar bootstrapping para intervalos de confianza

### 📋 Ejercicios

1. **Ejercicio 1**: Métricas de Forma (Skewness, Kurtosis)
2. **Ejercicio 2**: Estadística Robusta (MAD)
3. **Ejercicio 3**: Tests de Normalidad
4. **Ejercicio 4**: Bootstrapping para IC
5. **Ejercicio 5**: Análisis Completo de Distribución

---

## 🛠️ Setup

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis, shapiro, kstest
import warnings
warnings.filterwarnings('ignore')

# Cargar datos
ventas = pd.read_csv("/dbfs/FileStore/Laboratorio/Datasets/ventas.csv").head(1000)
print("✅ Datos cargados")
```

---

## 📊 Ejercicio 1: Asimetría y Curtosis

```python
columna = 'monto'

# Calcular métricas
skewness = skew(ventas[columna])
kurt = kurtosis(ventas[columna], fisher=True)

print(f"📊 Análisis de {columna}:\n")
print(f"Asimetría (Skewness): {skewness:.3f}")
if abs(skewness) < 0.5:
    print("  → ✅ Distribución simétrica")
elif skewness > 0.5:
    print("  → ⚠️ Asimetría positiva (cola derecha larga)")
else:
    print("  → ⚠️ Asimetría negativa (cola izquierda larga)")

print(f"\nCurtosis: {kurt:.3f}")
if abs(kurt) < 0.5:
    print("  → ✅ Similar a normal")
elif kurt > 0.5:
    print("  → ⚠️ Colas pesadas (más outliers)")
else:
    print("  → 📊 Colas ligeras (pocos outliers)")

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma
ventas[columna].hist(bins=30, ax=axes[0], edgecolor='black')
axes[0].axvline(ventas[columna].mean(), color='red', linestyle='--', label='Media')
axes[0].axvline(ventas[columna].median(), color='green', linestyle='--', label='Mediana')
axes[0].set_title(f"Distribución de {columna}")
axes[0].legend()

# Q-Q plot
stats.probplot(ventas[columna], dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot (vs Normal)")

plt.tight_layout()
plt.show()
```

---

## 🛡️ Ejercicio 2: Estadística Robusta (MAD)

```python
# Función MAD
def mad(data):
    median = np.median(data)
    return np.median(np.abs(data - median))

# Comparar métricas clásicas vs robustas
mean = ventas[columna].mean()
median = ventas[columna].median()
std = ventas[columna].std()
mad_value = mad(ventas[columna])

print("📊 Comparación: Métricas Clásicas vs Robustas\n")
print(f"Tendencia Central:")
print(f"  Media (clásica):     ${mean:,.2f}")
print(f"  Mediana (robusta):   ${median:,.2f}")
print(f"  Diferencia:          ${abs(mean - median):,.2f}")

print(f"\nDispersion:")
print(f"  Desv. Estándar (clásica): ${std:,.2f}")
print(f"  MAD (robusta):              ${mad_value:,.2f}")
print(f"  MAD escalado:               ${mad_value * 1.4826:,.2f}")  # Factor para comparar con std

# Detectar outliers con MAD
mod_z_score = 0.6745 * (ventas[columna] - median) / mad_value
outliers_mad = ventas[np.abs(mod_z_score) > 3.5]

print(f"\n⚠️ Outliers detectados con MAD: {len(outliers_mad)} ({len(outliers_mad)/len(ventas):.1%})")
```

---

## 📋 Ejercicio 3: Tests de Normalidad

```python
print("🔍 Tests de Normalidad\n" + "="*50)

# Test de Shapiro-Wilk
stat_shapiro, p_shapiro = shapiro(ventas[columna])
print(f"\n1. Shapiro-Wilk Test:")
print(f"   Estadística: {stat_shapiro:.4f}")
print(f"   p-value: {p_shapiro:.4f}")
if p_shapiro > 0.05:
    print("   → ✅ Distribución NORMAL (no rechazamos H0)")
else:
    print("   → ❌ Distribución NO NORMAL (rechazamos H0)")

# Test de Kolmogorov-Smirnov
data_normalized = (ventas[columna] - ventas[columna].mean()) / ventas[columna].std()
stat_ks, p_ks = kstest(data_normalized, 'norm')
print(f"\n2. Kolmogorov-Smirnov Test:")
print(f"   Estadística: {stat_ks:.4f}")
print(f"   p-value: {p_ks:.4f}")
if p_ks > 0.05:
    print("   → ✅ Distribución NORMAL")
else:
    print("   → ❌ Distribución NO NORMAL")

print("\n💡 Interpretación:")
print("  p > 0.05: No hay evidencia para rechazar normalidad")
print("  p ≤ 0.05: Evidencia de no normalidad")
print("\n🛠️ Si NO es normal → Usar estadística robusta o transformar datos")
```

---

## 🔄 Ejercicio 4: Bootstrapping para IC

```python
def bootstrap_ci(data, stat_func=np.mean, n_bootstrap=1000, confidence=0.95):
    """
    Calcula intervalo de confianza por bootstrapping
    """
    bootstrap_stats = []
    n = len(data)
    
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=n, replace=True)
        bootstrap_stats.append(stat_func(sample))
    
    alpha = (1 - confidence) / 2
    lower = np.percentile(bootstrap_stats, alpha * 100)
    upper = np.percentile(bootstrap_stats, (1 - alpha) * 100)
    
    return lower, upper, bootstrap_stats

# IC para media
lower_mean, upper_mean, dist_mean = bootstrap_ci(ventas[columna], stat_func=np.mean)

# IC para mediana
lower_median, upper_median, dist_median = bootstrap_ci(ventas[columna], stat_func=np.median)

print("🔄 Intervalos de Confianza (95%) por Bootstrapping\n" + "="*50)
print(f"\nMedia:")
print(f"  Estimado:  ${np.mean(ventas[columna]):,.2f}")
print(f"  IC 95%:    [${lower_mean:,.2f}, ${upper_mean:,.2f}]")

print(f"\nMediana:")
print(f"  Estimado:  ${np.median(ventas[columna]):,.2f}")
print(f"  IC 95%:    [${lower_median:,.2f}, ${upper_median:,.2f}]")

# Visualizar distribución bootstrap
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(dist_mean, bins=30, alpha=0.7, edgecolor='black')
axes[0].axvline(lower_mean, color='red', linestyle='--', label='IC 95%')
axes[0].axvline(upper_mean, color='red', linestyle='--')
axes[0].set_title("Bootstrap: Distribución de Media")
axes[0].set_xlabel("Media")
axes[0].legend()

axes[1].hist(dist_median, bins=30, alpha=0.7, color='green', edgecolor='black')
axes[1].axvline(lower_median, color='red', linestyle='--', label='IC 95%')
axes[1].axvline(upper_median, color='red', linestyle='--')
axes[1].set_title("Bootstrap: Distribución de Mediana")
axes[1].set_xlabel("Mediana")
axes[1].legend()

plt.tight_layout()
plt.show()
```

---

## 📊 Ejercicio 5: Reporte Completo de Distribución

```python
def analizar_distribucion(data, nombre_variable):
    """
    Análisis completo de una distribución
    """
    print(f"\n\n{'='*60}")
    print(f"REPORTE COMPLETO: {nombre_variable}")
    print(f"{'='*60}\n")
    
    # 1. Métricas básicas
    print("📊 1. ESTADÍSTICAS BÁSICAS")
    print(f"  N:              {len(data):,}")
    print(f"  Media:          ${np.mean(data):,.2f}")
    print(f"  Mediana:        ${np.median(data):,.2f}")
    print(f"  Desv. Est.:     ${np.std(data):,.2f}")
    print(f"  MAD:            ${mad(data):,.2f}")
    
    # 2. Percentiles
    print(f"\n📊 2. PERCENTILES")
    for p in [25, 50, 75, 90, 95, 99]:
        val = np.percentile(data, p)
        print(f"  P{p:2d}:            ${val:,.2f}")
    
    # 3. Forma
    print(f"\n📊 3. FORMA DE DISTRIBUCIÓN")
    sk = skew(data)
    kt = kurtosis(data, fisher=True)
    print(f"  Asimetría:     {sk:.3f}")
    print(f"  Curtosis:       {kt:.3f}")
    
    # 4. Normalidad
    print(f"\n📊 4. TEST DE NORMALIDAD")
    _, p_val = shapiro(data)
    print(f"  Shapiro p-val:  {p_val:.4f}")
    print(f"  Resultado:      {'Normal ✅' if p_val > 0.05 else 'No Normal ❌'}")
    
    # 5. Outliers
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers_count = np.sum((data < lower) | (data > upper))
    
    print(f"\n📊 5. OUTLIERS (IQR)")
    print(f"  Detectados:     {outliers_count} ({outliers_count/len(data):.1%})")
    print(f"  Rango normal:   [${lower:,.2f}, ${upper:,.2f}]")
    
    print(f"\n{'='*60}\n")

# Ejecutar análisis
analizar_distribucion(ventas['monto'].values, "Monto de Ventas")
```

---

## ✅ Resumen Final - Unidad 01 Completa

### 🎓 ¡Felicitaciones!

Completaste exitosamente **toda la Unidad 01: Análisis de Datos** del Material Complementario.

### 📚 Módulos Completados

✅ **Módulo 01**: EDA Avanzado con Profiling  
✅ **Módulo 02**: Manejo de Datos Faltantes y Outliers  
✅ **Módulo 03**: Análisis de Correlaciones  
✅ **Módulo 04**: Estadística Descriptiva Avanzada  

### 💡 Habilidades Adquiridas

**Análisis de Datos:**
* ✅ EDA automatizado (ydata-profiling, Sweetviz)
* ✅ Manejo de valores faltantes (MCAR, MAR, MNAR)
* ✅ Detección y tratamiento de outliers (IQR, Z-score, Isolation Forest)
* ✅ Análisis de correlaciones (Pearson, Spearman, Kendall)
* ✅ Estadística avanzada (skewness, kurtosis, MAD, bootstrapping)

**Herramientas:**
* ✅ pandas, numpy, scipy
* ✅ matplotlib, seaborn, plotly
* ✅ scikit-learn (imputers, Isolation Forest)
* ✅ ydata-profiling, sweetviz

### 🚀 Próximos Pasos

**En el curso:**
1. Aplica estos conceptos en los **Trabajos Prácticos**
2. Continúa con **Unidad 02: Visualización de Datos**
3. Usa este material como **referencia rápida**

**Fuera del curso:**
* 💻 Practica con datasets propios (Kaggle, UCI ML Repository)
* 📖 Profundiza con cursos especializados
* 👥 Comparte conocimiento con compañeros

---

### 🌟 Cita Final

> "En Dios confiamos, todos los demás traigan datos."  
> — W. Edwards Deming

---

**Universidad del Aconcagua**  
**Facultad de Ciencias Económicas y Jurídicas**  
**Licenciatura en Analítica de Negocios**  
**Mendoza, Argentina 🇦🇷**